# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [35]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
# from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI


from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


In [43]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'
openai = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=api_key,
)

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [44]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [45]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [46]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [47]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [48]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model="gemini-3.5-flash",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [49]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [50]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'enterprise page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [51]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [52]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.8-27B
Updated
about 4 hours ago
•
2
•
8.66k
meta-models/Muse-Glimmer-30B
Updated
3 days ago
•
165k
•
1.5k
Qwen/Qwen3.8-2.4T-A95B
Updated
2 days ago
•
3.83k
•
902
MiniMaxAI/MiniMax-H3
Updated
1 day ago
•
2M
•
3.91k
Lightricks/LTX-2.5
Updated
2 days ago
•
208k
•
831
Browse 2M+ models
Spaces
Running
on
Zero
MCP
Featured
2.52k
Qwen-Image-Edit-2511-LoRAs-Fast
🎃
2.52k
Demo of

In [53]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [54]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [55]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.8-27B\nUpdated\nabout 4 hours ago\n•\n2\n•\n8.67k\nmeta-models/Muse-Glimmer-30B\nUpdated\n3 days ago\n•\n

In [56]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gemini-3.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [57]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

Hugging Face is the world’s leading collaboration platform for machine learning. It is the central hub where the global AI community—encompassing researchers, developers, and organizations—comes together to build, share, and collaborate on models, datasets, and applications.

---

## The Home of Machine Learning

Hugging Face provides the infrastructure and tools that power the modern AI revolution. By democratizing access to state-of-the-art machine learning, we enable developers and businesses to build and deploy AI faster than ever before.

### Our Ecosystem by the Numbers
*   **2 Million+ Models:** From cutting-edge Large Language Models (LLMs) like Qwen and Muse-Glimmer to specialized vision and audio models.
*   **1 Million+ Applications (Spaces):** Live, interactive AI applications and demos ranging from video and music generators (like MiniMax H3) to AI text detectors.
*   **500,000+ Datasets:** High-quality training and evaluation datasets, including Fineweb and stack-v3-train, powering the next generation of AI training.

---

## Solutions for Businesses & Enterprise

Whether you are a startup or a Fortune 500 enterprise, Hugging Face provides robust solutions to build AI with confidence, security, and speed.

*   **Hugging Face Enterprise Support:** Gain direct access to our team of ML experts to accelerate your project roadmaps, optimize workloads, and secure your deployments.
*   **Inference Endpoints:** Deploy any of our 2M+ models onto secure, auto-scaling, and production-ready private infrastructure in just a few clicks.
*   **Storage Buckets:** Seamlessly store and manage large-scale data and model weights.
*   **Hugging Face PRO:** Power up your individual or small team workflows with upgraded compute, early access to new features, and enhanced platform capabilities.

---

## Why Invest in Hugging Face?

Hugging Face has established itself as the definitive "GitHub of Machine Learning." We sit at the very center of the AI ecosystem. 

*   **Unrivaled Network Effects:** As the default platform where researchers release their models and datasets first, our community-driven growth is unmatched.
*   **Enterprise-Grade Scalability:** We bridge the gap between open-source research and commercial application, offering secure, compliant, and scalable enterprise software solutions.
*   **Future-Proof Technology:** Supporting everything from LLMs and AI Agents to hardware optimizations and daily scientific paper discussions, Hugging Face is built to evolve with the rapid pace of AI innovation.

---

## Careers & Culture: Join the Movement

At Hugging Face, we believe that the future of AI should be open, collaborative, and built by a diverse, global community. 

### Why Join Us?
*   **Work on the Cutting Edge:** Engage daily with the world's most advanced AI research, models, and community creators.
*   **Collaborative & Open Culture:** We value open-source contribution, transparency, and a high-trust environment where builders are empowered to move fast and make an impact.
*   **Global Impact:** Your work will directly support millions of developers, researchers, and companies building technologies that shape the future of humanity.

*Explore our platform, build your first Space, or join our growing team today.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [58]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemini-3.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [59]:
stream_brochure("HuggingFace", "https://huggingface.co")

JSONDecodeError: Expecting ',' delimiter: line 55 column 4 (char 1217)

In [60]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

Hugging Face is the world's leading collaboration platform for machine learning. Known as the "Home of Machine Learning," Hugging Face is where the global AI community comes together to create, discover, and collaborate on models, datasets, and applications. 

Whether you are an enterprise looking to scale your AI capabilities, an investor seeking the epicenter of the AI revolution, or a builder looking to make a massive impact, Hugging Face is where the future is being built.

---

## What We Offer

Hugging Face provides an end-to-end ecosystem that empowers developers and enterprises to move from research to production seamlessly.

### The Open Hub
* **2 Million+ Models:** Access, share, and collaborate on state-of-the-art machine learning models across vision, language, audio, and more.
* **500,000+ Datasets:** Fuel your training runs with high-quality, community-curated datasets.
* **1 Million+ Applications (Spaces):** Build, host, and showcase interactive AI applications and demos—from video generation to custom music studios—powered by zero-configuration environments, agents, and custom hardware.

### Enterprise & Developer Solutions
To help businesses build fast and secure AI products, Hugging Face offers tailored solutions:
* **Team & Enterprise Plans:** Private collaborative environments with advanced security, compliance, and governance.
* **Hugging Face PRO:** Upgraded compute resources and features for individual power users.
* **Inference Endpoints:** Easily deploy models to secure, scalable, and production-ready APIs.
* **Enterprise Support:** Direct access to Hugging Face's expert team of machine learning engineers to accelerate your roadmap.
* **Storage Buckets & Hardware Solutions:** Optimized infrastructure for storing vast datasets and running heavy workloads.

---

## For Our Customers: Enterprise-Grade AI

From small startups to the world's largest organizations, Hugging Face is the trusted partner for modern AI development. Our platform enables teams to:
* **Collaborate Securely:** Host and collaborate on private models and datasets.
* **Avoid Vendor Lock-in:** Leverage open science and open-source models to maintain full control over your technology stack.
* **Deploy with Confidence:** Utilize optimized inference providers and dedicated support to scale your production AI seamlessly.

---

## For Investors: At the Core of the AI Revolution

Hugging Face occupies a unique and irreplaceable position in the AI ecosystem. We are the central repository and collaboration hub for almost every major breakthrough in open-source AI. 
* **Exponential Scale:** Hosting millions of models and applications, we are the default starting point for AI development globally.
* **Unrivaled Network Effects:** The platform effect of our community (spanning Discord, GitHub, and our Hub) ensures that when new AI research is published, it lands on Hugging Face first.
* **Monetization Engine:** Our enterprise subscriptions, inference endpoints, and compute solutions address the massive, growing demand for corporate AI adoption.

---

## Careers & Culture: Join the Movement

At Hugging Face, our mission is to democratize good machine learning. We believe in the power of open source, community, and collaboration to build a better future.

### Our Culture
* **Open & Collaborative:** We are deeply rooted in the open-source ethos. We share our work, support the community, and collaborate transparently.
* **Move Fast:** The AI landscape changes daily. We empower our team members to take ownership, experiment rapidly, and build tools that define the industry.
* **Community-First:** We prioritize building tools that help developers succeed, fostering an active ecosystem through our forums, Discord, Daily Papers, and learning resources.

### Why Join Us?
If you want to work on the cutting edge of technology alongside the creators of the libraries and tools that power the entire AI industry, Hugging Face is the place for you. Help us build the platform where the global machine learning community collaborates to shape the future of technology.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>